# RAISE HTML5 Semantification Demo

Owner: Siddharth Tripathi  
Repository: `semanticClimate/RAISE`  
Branch: `siddharth-semantification`  
Module folder: `RAISE_HTML5_Semantification`

Task: convert prepared annual-report block JSON into clean, traceable semantic HTML5.

Upload `target_blocks.json`. Do not upload a PDF; PDF parsing and filtering happen before this module.

## Upload Prepared Block JSON

This cell opens the Colab file picker, records the uploaded JSON filename in `input_filename`, and confirms the selected input. It is required because the conversion stage reads structured annual-report blocks from that filename. The expected input is JSON produced by the earlier PDF extraction and filtering stage, not a PDF.

In [ ]:
from google.colab import files

uploaded = files.upload()
input_filename = next(iter(uploaded))
print(f"Uploaded JSON file: {input_filename}")

## Install Runtime Dependencies

This cell installs the Python libraries required for HTML parsing, validation, data models, templates, and command-line support. It is included so a fresh Colab runtime has the same required dependencies before the semantification code runs.

In [ ]:
!pip install -q beautifulsoup4 lxml jinja2 pydantic jsonschema typer rich

## Install the RAISE Module

This cell first tries to install `RAISE_HTML5_Semantification` from the `siddharth-semantification` branch of the `semanticClimate/RAISE` repository and prints the full pip output. If GitHub access fails because the repository is private, the next cell lets you upload a local ZIP and installs the package from that upload.

In [ ]:
# Try GitHub first. If the private repository is not accessible, the next cell installs from an uploaded ZIP.

import subprocess
import sys

GITHUB_PACKAGE = "git+https://github.com/semanticClimate/RAISE.git@siddharth-semantification#subdirectory=RAISE_HTML5_Semantification"

cmd = [
    sys.executable,
    "-m",
    "pip",
    "install",
    "--upgrade",
    "--no-cache-dir",
    GITHUB_PACKAGE,
]

print("Running:")
print(" ".join(cmd))

result = subprocess.run(cmd, text=True, capture_output=True)

print("
STDOUT:")
print(result.stdout or "(empty)")
print("
STDERR:")
print(result.stderr or "(empty)")

PACKAGE_INSTALLED = result.returncode == 0
if PACKAGE_INSTALLED:
    print("Installed RAISE_HTML5_Semantification from semanticClimate/RAISE.")
else:
    print("GitHub install failed. If this repository is private, run the next cell and upload a ZIP of the module.")

## Private Repository Fallback

Run this cell only if the GitHub install cell reports failure. Upload a ZIP containing either `RAISE_HTML5_Semantification/RAISE_HTML5_Semantification` or the larger RAISE folder. The cell finds the semantification package and installs it locally with a normal `pip install`.

In [ ]:
if not globals().get("PACKAGE_INSTALLED", False):
    from google.colab import files
    import shutil
    import subprocess
    import sys
    import zipfile
    from pathlib import Path

    uploaded = files.upload()
    zip_name = next(iter(uploaded))
    extract_dir = Path("/content/raise_html5_upload")

    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_name, "r") as archive:
        archive.extractall(extract_dir)

    module_root = None
    for project_file in extract_dir.rglob("pyproject.toml"):
        if 'name = "raise-html5-semantification"' in project_file.read_text(encoding="utf-8"):
            module_root = project_file.parent
            break

    if module_root is None:
        raise FileNotFoundError("Could not find raise-html5-semantification pyproject.toml in the uploaded ZIP.")

    install_commands = []
    if shutil.which("uv"):
        install_commands.append(["uv", "pip", "install", "--system", "--upgrade", "--no-cache-dir", "-e", str(module_root)])
    install_commands.append([sys.executable, "-m", "pip", "install", "--upgrade", "--no-cache-dir", str(module_root)])

    last_result = None
    package_imported = False
    for cmd in install_commands:
        print("Installing from uploaded ZIP:")
        print(" ".join(cmd))
        last_result = subprocess.run(cmd, text=True, capture_output=True)
        print()
        print("STDOUT:")
        print(last_result.stdout or "(empty)")
        print()
        print("STDERR:")
        print(last_result.stderr or "(empty)")
        if last_result.returncode == 0:
            try:
                import raise_html5_semantification  # noqa: F401
            except ModuleNotFoundError:
                print("Install completed, but the package is not importable yet. Trying the next fallback.")
                continue
            package_imported = True
            break

    if last_result is None or last_result.returncode != 0 or not package_imported:
        raise RuntimeError(f"Local ZIP install failed with exit code {getattr(last_result, 'returncode', 'unknown')}")

    PACKAGE_INSTALLED = True
    print("Installed RAISE_HTML5_Semantification from uploaded ZIP.")
else:
    print("Package already installed from GitHub; skipping ZIP fallback.")

## Identify the Colab Runtime

This cell checks the Colab environment for a TPU address and queries `nvidia-smi` for an NVIDIA GPU model such as a T4. It is included to report the active accelerator clearly; the semantification workflow remains deterministic when Colab provides only a CPU.

In [ ]:
import os
import platform
import subprocess

def detect_gpu():
    try:
        result = subprocess.run(
            ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
            capture_output=True, text=True, check=True,
        )
        return result.stdout.strip()
    except (FileNotFoundError, subprocess.CalledProcessError):
        return ""

gpu_name = detect_gpu()
runtime_info = {
    "platform": platform.platform(),
    "colab_tpu_addr": os.environ.get("COLAB_TPU_ADDR", ""),
    "gpu_name": gpu_name,
}

if runtime_info["colab_tpu_addr"]:
    print(f"TPU runtime detected: {runtime_info['colab_tpu_addr']}")
elif runtime_info["gpu_name"]:
    print(f"GPU runtime detected: {runtime_info['gpu_name']}")
else:
    print("CPU runtime detected. HTML5 semantification is CPU-safe and deterministic.")

runtime_info

## Import the Installed Repository Engine

This cell imports the production semantification API from the `raise_html5_semantification` package installed by the GitHub cell. It loads the repository functions for JSON validation, semantic-profile learning, section construction, HTML rendering, output validation, and end-to-end report generation. Printing the package path confirms which installed module Colab will execute. The smoke tests and report conversion below call these imports directly; no separate notebook engine is used.

In [ ]:
import json
from pathlib import Path
from tempfile import TemporaryDirectory

from bs4 import BeautifulSoup
import raise_html5_semantification
from raise_html5_semantification.html_writer import render_document
from raise_html5_semantification.loader import load_blocks
from raise_html5_semantification.main import build_report
from raise_html5_semantification.profiler import learn_semantic_profile
from raise_html5_semantification.section_builder import build_section_tree
from raise_html5_semantification.validator import validate_html_string

package_path = Path(raise_html5_semantification.__file__).resolve()
print(f"Using installed repository engine: {package_path}")

## JSON Robustness Smoke Tests

This cell creates three small annual-report-like JSON cases and sends each case through the same extraction, normalization, classification, rendering, and traceability logic used by the conversion workflow. The cases cover missing block IDs, nested page structures, alternate field names, headings, ordered and unordered lists, clear tables, uncertain table candidates, and low-confidence OCR text.

The final assertion requires every case to produce enough HTML elements carrying an ID, source-block reference, and semantic role. This cell is included as an early integrity check: if a core transformation rule breaks, execution stops before the uploaded report is converted.

In [ ]:
SMOKE_CASES = {
    "minimal_missing_ids": [
        {"text": "PUBLICATIONS", "font_size": 19, "is_bold": True, "confidence": 0.96},
        {"text": "The faculty published journal articles and conference papers.", "confidence": 0.85},
        {"text": "• Journal articles: 45\n• Conference papers: 62", "confidence": 0.82},
        {"text": "possibly unreadable handwritten annotation", "confidence": 0.18},
    ],
    "nested_annual_report": {
        "document": {
            "pages": [
                {
                    "pageNumber": 4,
                    "elements": [
                        {"content": "DEPARTMENT OF COMPUTER SCIENCE", "page": 4, "fontSize": 20, "bold": True, "role": "heading", "probability": 0.99},
                        {"content": "Research Grants and Collaboration", "page": 4, "fontSize": 15, "bold": True, "section": "Grants", "probability": 0.94},
                        {"content": "1. DST project continued\n2. Industry collaboration signed", "page": 4, "probability": 0.91},
                        {"content": "Grant Scheme | Amount | PI\nSERB | 1250000 | Dr. Rao", "page": 5, "label": "table-candidate", "probability": 0.87},
                        {"content": "smudged footer possible duplicate text", "page": 5, "probability": 0.21},
                    ],
                }
            ]
        }
    },
    "table_and_outreach": {
        "content_blocks": [
            {"uid": "awards-heading", "raw_text": "FACULTY AWARDS", "font_size": 17, "is_bold": True, "category": "title", "confidence": 0.97},
            {"uid": "awards-table-clear", "block_type": "table", "table_rows": [["Name", "Award"], ["Dr. Iyer", "Best Researcher"]], "confidence": 0.93},
            {"uid": "awards-table-unclear", "value": "Name Award Year maybe columns broken", "block_type": "table-candidate", "confidence": 0.52},
            {"uid": "outreach-list", "value": "- Workshop for schools\n- Conference tutorial", "confidence": 0.89},
        ]
    },
}


def semantify_raw_for_smoke(raw, name):
    with TemporaryDirectory() as temp_dir:
        input_path = Path(temp_dir) / f"{name}.json"
        input_path.write_text(json.dumps(raw), encoding="utf-8")

        blocks = load_blocks(input_path)
        profile = learn_semantic_profile(blocks, source_name=name)
        nodes = build_section_tree(blocks, profile=profile)
        report_html = render_document(nodes, title=f"Smoke test: {name}")
        summary = validate_html_string(report_html, expected_blocks=len(blocks))
        return {
            "name": name,
            "ok": summary.ok,
            "blocks": len(blocks),
            "traceable_elements": summary.traceable_element_count,
            "profile": profile.model_dump(),
        }


smoke_results = [semantify_raw_for_smoke(raw, name) for name, raw in SMOKE_CASES.items()]
assert all(result["ok"] for result in smoke_results), smoke_results
smoke_results

## Convert the Uploaded JSON

This cell calls the installed repository's `build_report()` API with `input_filename`. The package validates and normalizes the JSON, learns a semantic profile, builds the document structure, renders traceable accessible HTML5, and writes the HTML, source map, section map, AI-ready chunks, input-quality report, validation report, and semantic profile. Reading the review outputs back into memory prepares them for the next cell.

In [ ]:
semantic_html_path = "report.html"
source_map_path = "source_map.json"
section_map_path = "section_map.json"
ai_chunks_path = "ai_chunks.json"
validation_report_path = "validation_report.json"
input_quality_report_path = "input_quality_report.json"
semantic_profile_path = "semantic_profile.json"

report_html = build_report(
    input_path=input_filename,
    output_path=semantic_html_path,
    validation_output_path=validation_report_path,
    profile_output_path=semantic_profile_path,
    source_map_output_path=source_map_path,
    section_map_output_path=section_map_path,
    ai_chunks_output_path=ai_chunks_path,
    input_quality_output_path=input_quality_report_path,
)
validation_report = json.loads(Path(validation_report_path).read_text(encoding="utf-8"))
input_quality_report = json.loads(Path(input_quality_report_path).read_text(encoding="utf-8"))
semantic_profile = json.loads(Path(semantic_profile_path).read_text(encoding="utf-8"))
validation_report

## Review the Generated Results

This cell prints the validation result, upstream input-quality report, and learned semantic profile, then displays `report.html` inside the notebook. It is included to verify the conversion status and visually review the semantic HTML before downloading the outputs.

In [ ]:
from IPython.display import IFrame, display

print("Validation report:")
print(json.dumps(validation_report, indent=2))

print("Input quality report:")
print(json.dumps(input_quality_report, indent=2))

print("Semantic profile:")
print(json.dumps(semantic_profile, indent=2))

display(IFrame("report.html", width="100%", height=650))

## Download the Deliverables

This cell packages the complete generated deliverable set into a ZIP, then downloads the HTML, source map, section map, AI-ready chunks, validation report, input-quality report, and full ZIP through Colab.

In [ ]:
from zipfile import ZIP_DEFLATED, ZipFile

download_paths = [
    semantic_html_path,
    source_map_path,
    section_map_path,
    ai_chunks_path,
    validation_report_path,
    input_quality_report_path,
]
full_zip_path = "raise_html5_semantification_outputs.zip"
with ZipFile(full_zip_path, "w", compression=ZIP_DEFLATED) as archive:
    for output_path in [*download_paths, semantic_profile_path]:
        archive.write(output_path, arcname=Path(output_path).name)

for output_path in [*download_paths, full_zip_path]:
    files.download(output_path)